Generate a chart for a validation song and compare it with the ground truth

Runs top to bottom: setup, load validation data, load model, generate, export.

In [5]:
from pathlib import Path

from tracers.project import setup_project
from tracers.datasets.factory import build_datasets, build_loader_for_model
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_np"

experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
batch_size = setup["batch_size"]
device = setup["device"]

In [6]:
audio_dir, beatmap_dir, train_dataset, val_dataset, test_dataset, train_ids, val_ids, test_ids = build_datasets(
    PROJECT_ROOT,
    segment_frames=segment_frames,
    seed=42,
)

train_loader = build_loader_for_model(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)
val_loader = build_loader_for_model(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)

In [7]:
from tracers.pipelines.evaluate import run_evaluation

trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=str(PROJECT_ROOT / "notebooks" / "checkpoints" / "best.pt"),
    train_loader=None,
    valid_loader=None,
)

CHECKPOINT = PROJECT_ROOT / "notebooks/checkpoints/best.pt"

Loaded (epoch 5, best val loss 0.1488)


In [10]:
run_evaluation(trainer, val_loader, checkpoint=str(CHECKPOINT), name="VAL")

Loaded (epoch 5, best val loss 0.1488)


Validation: 100%|██████████| 151/151 [00:24<00:00,  6.04it/s]


VAL (25s)
Loss   | Total: 0.1488 | NP: 0.0129 | NS: 0.0588
Onset  | P: 0.7563 | R: 0.7442 | F1: 0.7502
Action | P: 0.8723 | R: 0.8408 | F1: 0.8563
Active Exact Accuracy: 0.3744
GT Active Rate: 4.37% | Predicted Active Rate: 4.21%


In [1]:
from pathlib import Path
import runpy

PROJECT_ROOT = Path.cwd().parent
script_path = PROJECT_ROOT / 'notebooks' / 'eval_autoosu_baseline.py'
module = runpy.run_path(str(script_path))
module['evaluate_autoosu_checkpoint'](root=PROJECT_ROOT)



AutoOsu baseline (osu_model_v2.pt)
Loss   | Total: 0.0787 | NP: 0.0050 | NS: 0.0436
Onset  | P: 0.8930 | R: 0.9260 | F1: 0.9092
Action | P: 0.9441 | R: 0.9301 | F1: 0.9370
Active Exact Accuracy: 0.4389
GT Active Rate: 4.35% | Predicted Active Rate: 4.28%


{'losses': {'total': 0.07873232965796963,
  'np': 0.005016313829140294,
  'ns': 0.043618133069191004},
 'metrics': {'onset_precision': 0.8930081556245009,
  'onset_recall': 0.9260425716768028,
  'onset_f1': 0.9092254070228519,
  'action_precision': 0.9440831174557681,
  'action_recall': 0.9300879669852302,
  'action_f1': 0.937033288656692,
  'active_exact_accuracy': 0.4389118158123371,
  'gt_active_rate': 0.04345863836600617,
  'predicted_active_rate': 0.04281440464131461}}